# Fairlearn: is the model fair, to whom, and what can we do about it?

The DVC and MLflow lessons kept showing one uncomfortable number: the model catches
**fewer malnourished girls than boys**. This notebook asks properly:

1. Who gets flagged, and who gets missed?
2. Which definition of "fair" matters for a screening tool?
3. Are the gaps real, or noise from small groups?
4. *Why* are girls missed?
5. What can we change, and what does it cost?

**Setting.** A clinic can follow up 20% of visits. The model ranks children by risk and
the top 20% are flagged. The costly error is a **miss**: a child who becomes acutely
malnourished but wasn't flagged. A false alarm costs a follow-up visit.

Fairlearn's core tool is **`MetricFrame`**: any metric, computed overall *and* per group.

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.insert(0, "../03-mlflow")       # reuse the MLflow lesson's data/model/evaluation helpers
import emr_model as em

import numpy as np
import pandas as pd
from fairlearn.metrics import (MetricFrame, equal_opportunity_difference, false_negative_rate,
                               selection_rate, true_positive_rate)
from sklearn.metrics import precision_score, roc_auc_score

pd.set_option("display.width", 160, "display.precision", 3)
RAW = Path("data/raw")                   # this lesson's own data folder

# Train on 2024 (as in the other lessons), evaluate on a full YEAR after deployment (2025):
# 24,000 visits, so small groups still have enough cases to say something.
em.simulate("2024-01", 12, out=RAW)
em.simulate("2025-01", 12, out=RAW)
train = em.load_months("2024-01", "2024-12", raw=RAW)
evalset = em.load_months("2025-01", "2025-12", raw=RAW)

model = em.build_model("logistic").fit(train[em.FEATURES], train[em.LABEL])
y = evalset[em.LABEL].to_numpy()                                  # what happened
score = model.predict_proba(evalset[em.FEATURES])[:, 1]           # predicted risk
flag = (score >= np.quantile(score, 0.8)).astype(int)             # top 20% flagged
print(f"{len(evalset)} visits, {y.mean():.1%} developed GAM; ROC-AUC {roc_auc_score(y, score):.3f}; "
      f"flagged {flag.mean():.0%}; sensitivity {flag[y == 1].mean():.3f}")

24000 visits, 11.0% developed GAM; ROC-AUC 0.814; flagged 20%; sensitivity 0.577


## 1. Who gets flagged, and who gets missed?

In [2]:
metrics = {
    "prevalence":  lambda yt, yp: yt.mean(),      # share who developed GAM
    "alert_rate":  selection_rate,                # share flagged
    "sensitivity": true_positive_rate,            # of those who developed GAM, share flagged
    "miss_rate":   false_negative_rate,           # of those who developed GAM, share MISSED
    "ppv":         precision_score,               # of those flagged, share who developed GAM
    "n":           lambda yt, yp: len(yt),
}
by_sex = MetricFrame(metrics=metrics, y_true=y, y_pred=flag, sensitive_features=evalset["sex"])
print("overall:", by_sex.overall.round(3).to_dict())
by_sex.by_group

overall: {'prevalence': 0.11, 'alert_rate': 0.2, 'sensitivity': 0.577, 'miss_rate': 0.423, 'ppv': 0.319, 'n': 24000.0}


,prevalence,alert_rate,sensitivity,miss_rate,ppv,n
sex,,,,,,
F,0.111,0.191,0.541,0.459,0.317,11685.0
M,0.109,0.209,0.612,0.388,0.321,12315.0


In [3]:
print("largest gap between groups (difference):")
print(by_sex.difference().round(3).to_dict())
print("equal_opportunity_difference:", round(equal_opportunity_difference(y, flag, sensitive_features=evalset["sex"]), 3))

largest gap between groups (difference):
{'prevalence': 0.002, 'alert_rate': 0.019, 'sensitivity': 0.071, 'miss_rate': 0.071, 'ppv': 0.004, 'n': 630.0}
equal_opportunity_difference: 0.071


**Read it like a clinician.** Girls and boys develop malnutrition at the same rate
(~11%), but of the children who do, the model flags **54% of girls vs 61% of boys**.
Out of every 100 malnourished girls, about 7 more are missed than among boys. The
flags are equally reliable (PPV ~0.32 for both), so the problem is who gets flagged
in the first place.

## 2. Which definition of "fair"?

"Fair" can mean several things, and they are not compatible:

| Definition | What must be equal across groups | In this clinic it means... | Fairlearn |
|---|---|---|---|
| **Demographic parity** | alert rate | the same share of girls and boys get follow-up | `demographic_parity_difference` |
| **Equal opportunity** | sensitivity | a malnourished girl is as likely to be caught as a malnourished boy | `equal_opportunity_difference` |
| **Equalized odds** | sensitivity *and* false-alarm rate | as above, plus equal false alarms | `equalized_odds_difference` |
| **Predictive parity** | PPV | a flag means the same risk whoever gets it | `MetricFrame` with `precision_score` |

When groups have **different base rates**, you cannot satisfy all of these at once.
Regions show why:

In [4]:
by_region = MetricFrame(metrics=metrics, y_true=y, y_pred=flag, sensitive_features=evalset["region"])
by_region.by_group.sort_values("prevalence")

,prevalence,alert_rate,sensitivity,miss_rate,ppv,n
region,,,,,,
Addis Ababa,0.026,0.021,0.154,0.846,0.186,2030.0
Oromia,0.099,0.165,0.496,0.504,0.296,7285.0
Amhara,0.109,0.195,0.497,0.503,0.278,5147.0
Sidama,0.111,0.153,0.509,0.491,0.368,1971.0
SNNP,0.114,0.214,0.613,0.387,0.326,3685.0
Tigray,0.134,0.268,0.685,0.315,0.343,1613.0
Afar,0.196,0.412,0.801,0.199,0.381,1076.0
Somali,0.212,0.487,0.794,0.206,0.346,1193.0


Afar and Somali children develop malnutrition far more often than children in Addis
Ababa. Equal **alert rates** (demographic parity) would mean flagging well children in
Addis and turning away sick children in Afar: clearly wrong here.

But look at **sensitivity**: only ~15% of malnourished children in Addis Ababa are
flagged, against ~80% in Afar and Somali. One national threshold sees Addis children as
low-risk, so the few who do become malnourished are mostly missed. By the
equal-opportunity standard, this is a bigger gap than the one between girls and boys
(exercise 1). For a screening tool
the usual priority is **equal opportunity**: a malnourished child should have the same
chance of being caught wherever and whoever they are. Different alert rates are fine,
and expected, when need differs.

That is a **choice**, and it should be written down with the people who own the
programme. In hiring (the resume use case), the answer is often different.

## 3. Intersections, and how sure are we?

Gaps can hide inside combinations of groups. Small groups also produce noisy numbers.
`MetricFrame` can bootstrap confidence intervals: resample the data many times and see
how much the metric moves.

In [5]:
def sensitivity_with_ci(data, scores, label=""):
    yy = data[em.LABEL].to_numpy()
    ff = (scores >= np.quantile(scores, 0.8)).astype(int)
    mf = MetricFrame(metrics={"sensitivity": true_positive_rate}, y_true=yy, y_pred=ff,
                     sensitive_features=data[["sex", "age_band"]],
                     n_boot=200, ci_quantiles=[0.05, 0.95], random_state=0)   # 90% interval
    lo, hi = mf.by_group_ci
    cases = data[data[em.LABEL] == 1].groupby(["sex", "age_band"], observed=True).size()
    return pd.DataFrame({"sensitivity": mf.by_group["sensitivity"], "low": lo["sensitivity"],
                         "high": hi["sensitivity"], "GAM cases": cases})

two_months = em.load_months("2025-01", "2025-02", raw=RAW)
print("Evaluated on 2 months (4,000 visits):")
display(sensitivity_with_ci(two_months, model.predict_proba(two_months[em.FEATURES])[:, 1]).round(2))
print("Evaluated on 12 months (24,000 visits):")
sensitivity_with_ci(evalset, score).round(2)

Evaluated on 2 months (4,000 visits):


sensitivity   low  high  GAM cases
sex age_band                                    
F   0-5m             0.22  0.07  0.35         23
    24-59m           0.54  0.40  0.65         39
    6-23m            0.60  0.52  0.69        102
M   0-5m             0.74  0.56  0.89         23
    24-59m           0.41  0.29  0.54         37
    6-23m            0.69  0.62  0.77        110

Evaluated on 12 months (24,000 visits):


sensitivity   low  high  GAM cases
sex age_band                                    
F   0-5m             0.36  0.30  0.43        197
    24-59m           0.54  0.50  0.59        351
    6-23m            0.59  0.56  0.62        754
M   0-5m             0.45  0.40  0.51        202
    24-59m           0.51  0.47  0.56        285
    6-23m            0.68  0.66  0.71        861

- With **2 months**, the youngest children show a dramatic gap: girls ~0.22 vs boys
  ~0.74. It comes from **23 cases each**, and the intervals are huge.
- With **12 months**, that "gap" shrinks to ~0.36 vs ~0.45, with overlapping intervals.
  Most of the dramatic 2-month result was noise.
- The gap that holds up is at **6–23 months** (girls ~0.59 vs boys ~0.68, intervals
  apart), the age when malnutrition risk peaks. That's where it matters most.
- In 24–59 months, girls and boys overlap: no evidence of a gap there.

Report per-group metrics **with intervals and case counts**, or people will act on noise.

> **PAUSE (2–3 min).** Girls and boys develop GAM at the same rate, yet the model misses
> more girls. Where could that come from? (Hint: the data notebook, section 3.)

## 4. Why are girls missed?

Recall the data notebook: at the same weight-for-height, **girls' MUAC reads lower**
than boys'. What does the model make of that?

In [6]:
train["muac_band"] = pd.cut(train["muac_cm"], [12.5, 13, 13.5, 14, 20])
print("GAM within 90 days, by MUAC band and sex (training data):")
display(train.groupby(["muac_band", "sex"], observed=True)[em.LABEL].mean().unstack().round(3))

coef = pd.Series(model[-1].coef_[0], index=model[0].get_feature_names_out())
print("model coefficients for sex:", coef.filter(like="sex").round(3).to_dict())

GAM within 90 days, by MUAC band and sex (training data):


sex,F,M
muac_band,,
"(12.5, 13.0]",0.245,0.278
"(13.0, 13.5]",0.119,0.152
"(13.5, 14.0]",0.061,0.068
"(14.0, 20.0]",0.025,0.025


model coefficients for sex: {'cat__sex_F': -0.528, 'cat__sex_M': -0.429}


At the **same MUAC**, girls develop GAM *less* often than boys, because a girl's MUAC
of 13 cm reflects better nutrition than a boy's 13 cm. The model learns this through
the `sex` feature, effectively "being a girl lowers risk" (`sex_F` has the more
negative coefficient). Girls who *do* become malnourished then rank lower and miss the
cut more often.

**The measurement differs by group, and the model inherits it.** No-one wrote a biased
line of code. This is the most common way unfairness enters clinical models.

What if the model doesn't see `sex` at all ("fairness through unawareness")?

In [7]:
features_no_sex = [f for f in em.FEATURES if f != "sex"]
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

model_no_sex = make_pipeline(
    ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), em.NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), [c for c in em.CATEGORICAL if c != "sex"]),
    ]),
    LogisticRegression(max_iter=1000),
).fit(train[features_no_sex], train[em.LABEL])
score_no_sex = model_no_sex.predict_proba(evalset[features_no_sex])[:, 1]


def summarise(name, scores, flags):
    # one row of the comparison table: overall quality, workload, and the girls/boys gap
    s = MetricFrame(metrics=true_positive_rate, y_true=y, y_pred=flags.astype(int),
                    sensitive_features=evalset["sex"]).by_group
    return {"approach": name, "roc_auc": roc_auc_score(y, scores), "alert_rate": flags.mean(),
            "sensitivity": flags[y == 1].mean(), "girls": s["F"], "boys": s["M"],
            "gap (boys-girls)": s["M"] - s["F"]}


top20 = lambda s: s >= np.quantile(s, 0.8)
results = [summarise("baseline", score, top20(score)),
           summarise("no sex feature", score_no_sex, top20(score_no_sex))]
pd.DataFrame(results).set_index("approach").round(3)

,roc_auc,alert_rate,sensitivity,girls,boys,gap (boys-girls)
approach,,,,,,
baseline,0.814,0.2,0.577,0.541,0.612,0.071
no sex feature,0.815,0.2,0.583,0.565,0.601,0.036


Dropping `sex` halves the gap at no cost in ROC-AUC. It works *here* because `sex`
itself carried the problem. Often unawareness doesn't help: other features act as
proxies (a feature correlated with sex can carry the same pattern), and sometimes the
sensitive feature is what lets a model treat groups *correctly*. Always measure; never
assume.

### Would a more flexible model fix it?
The model already *does* adjust for sex (`sex_F` above). Two natural next ideas: let the adjustment depend on
the measurements (a logistic model with **sex × every number** interactions), or use a model that finds such
interactions itself (**gradient-boosted trees**). Same data, same 20% capacity:

In [8]:
def with_interactions(d):
    d = d.copy()
    for c in em.NUMERIC:
        d[f"{c}_xF"] = d[c] * (d["sex"] == "F")      # a separate slope for girls on every number
    return d

num_int = em.NUMERIC + [f"{c}_xF" for c in em.NUMERIC]
tr_i, ev_i = with_interactions(train), with_interactions(evalset)
cols_i = em.FEATURES + [f"{c}_xF" for c in em.NUMERIC]
model_int = make_pipeline(
    ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), num_int),
        ("cat", OneHotEncoder(handle_unknown="ignore"), em.CATEGORICAL),
    ]),
    LogisticRegression(max_iter=2000),
).fit(tr_i[cols_i], tr_i[em.LABEL])
score_int = model_int.predict_proba(ev_i[cols_i])[:, 1]

gbm3 = em.build_model("gbm", max_depth=3).fit(train[em.FEATURES], train[em.LABEL])
gbm6 = em.build_model("gbm", max_depth=6).fit(train[em.FEATURES], train[em.LABEL])
score_gbm3 = gbm3.predict_proba(evalset[em.FEATURES])[:, 1]
score_gbm6 = gbm6.predict_proba(evalset[em.FEATURES])[:, 1]

pd.DataFrame(results + [summarise("logistic + sex interactions", score_int, top20(score_int)),
                        summarise("boosted trees, depth 3", score_gbm3, top20(score_gbm3)),
                        summarise("boosted trees, depth 6", score_gbm6, top20(score_gbm6))]).set_index("approach").round(3)

,roc_auc,alert_rate,sensitivity,girls,boys,gap (boys-girls)
approach,,,,,,
baseline,0.814,0.2,0.577,0.541,0.612,0.071
no sex feature,0.815,0.2,0.583,0.565,0.601,0.036
logistic + sex interactions,0.813,0.2,0.577,0.542,0.610,0.068
"boosted trees, depth 3",0.808,0.2,0.562,0.534,0.588,0.054
"boosted trees, depth 6",0.801,0.2,0.552,0.528,0.575,0.047


More flexibility does **not** remove the gap: interactions leave it where it was, trees shrink it a little
(and lose a little ROC-AUC). The model is not too simple. The problem is in the **data and the label**: the label
is partly defined by MUAC, which reads lower in girls at the same nutrition, and nothing in the record says
whether a girl's low MUAC is poor nutrition or just how girls measure. No model can recover what the data does
not contain. That is why the fixes are about **decisions** (the threshold, the features you allow, who reviews
the flags), not about a cleverer model.

> **PAUSE (3 min).** The clinic can follow up 20% of visits, no more. How would you close
> the girls/boys gap without exceeding that?

## 5. Mitigation within the clinic's capacity

The clinic can follow up 20% of visits, no more. One direct fix: **separate thresholds
for girls and boys**, chosen so both groups get the same sensitivity, with the total
number of flags kept within capacity.

In [9]:
def equal_sensitivity_flags(scores, groups, capacity=0.20):
    """Per-group thresholds giving every group the same sensitivity, as high as capacity allows.

    Thresholds are chosen on known outcomes. In practice you set them on a past period
    and apply them going forward (exercise 2).
    """
    best = None
    for target in np.arange(0.40, 0.90, 0.005):              # try ever-higher sensitivity targets
        flags = np.zeros(len(scores), bool)
        for g in np.unique(groups):
            in_g = groups == g
            cases = np.sort(scores[in_g & (y == 1)])[::-1]   # risk scores of the group's GAM cases
            cut = cases[int(np.ceil(target * len(cases))) - 1]
            flags[in_g] = scores[in_g] >= cut
        if flags.mean() > capacity:
            break                                             # over capacity: keep the previous target
        best = flags
    return best


flags_eq = equal_sensitivity_flags(score, evalset["sex"].to_numpy())
results.append(summarise("per-sex thresholds", score, flags_eq))
pd.DataFrame(results).set_index("approach").round(3)

,roc_auc,alert_rate,sensitivity,girls,boys,gap (boys-girls)
approach,,,,,,
baseline,0.814,0.200,0.577,0.541,0.612,0.071
no sex feature,0.815,0.200,0.583,0.565,0.601,0.036
per-sex thresholds,0.814,0.198,0.580,0.581,0.580,-0.001


Equal sensitivity for girls and boys, within capacity, and overall sensitivity is no
worse. The model didn't change, only the decision rule on top of it.

### Fairlearn's general tool: `ThresholdOptimizer`

Fairlearn automates group-specific thresholds. Used with its defaults:

In [10]:
from fairlearn.postprocessing import ThresholdOptimizer

optimizer = ThresholdOptimizer(estimator=model, prefit=True, predict_method="predict_proba",
                               constraints="true_positive_rate_parity",   # = equal opportunity
                               objective="balanced_accuracy_score")
optimizer.fit(train[em.FEATURES], train[em.LABEL], sensitive_features=train["sex"])
flags_to = optimizer.predict(evalset[em.FEATURES], sensitive_features=evalset["sex"], random_state=0).astype(bool)
results.append(summarise("ThresholdOptimizer (defaults)", score, flags_to))
pd.DataFrame(results).set_index("approach").round(3)

,roc_auc,alert_rate,sensitivity,girls,boys,gap (boys-girls)
approach,,,,,,
baseline,0.814,0.200,0.577,0.541,0.612,0.071
no sex feature,0.815,0.200,0.583,0.565,0.601,0.036
per-sex thresholds,0.814,0.198,0.580,0.581,0.580,-0.001
ThresholdOptimizer (defaults),0.814,0.332,0.762,0.746,0.777,0.032


Sensitivity jumps, but **a third of all visits** are flagged: its objective (balanced
accuracy) knows nothing about the clinic's 20% capacity. The gap also shrinks less
than with our hand-made rule. A fairness tool optimises what you tell it to. Here the
real constraint is capacity, so the simple rule that encodes it does better.

Fairlearn also has **reductions** (e.g. `ExponentiatedGradient`) that build fairness
into training. They produce yes/no decisions rather than risk scores, so they fit
"approve / reject" settings better than "rank and follow up the top 20%". We tried
it; it isn't useful for this ranking setup.

## 6. Decide, document, monitor

| Approach | Gap (boys − girls) | Cost | Questions to ask |
|---|---|---|---|
| Baseline | ~0.07 | none | Is a 7-point gap in missed girls acceptable? |
| Drop `sex` | ~0.04 | none here | Does it hold on new data? Proxies? |
| Per-sex thresholds | ~0 | uses sex at decision time | Acceptable clinically? (Growth standards are already sex-specific.) |
| Fix the measurement | ? | programme change | Sex-specific MUAC cut-offs? Use WHZ where possible? |

- **Write down the definition** (equal opportunity), the groups (sex, age band, region),
  and the tolerance (e.g. gap < 0.03), before looking at models.
- **Make it a gate.** The MLflow lesson's promotion rule already refuses a challenger that
  lowers girls' or boys' sensitivity; per-group metrics are logged with every run.
- **Monitor it.** Gaps can appear later, when the population or the measurement changes
  (Evidently lesson). Outcomes arrive ~90 days late, so fairness monitoring lags too.
- **The resume-matching use case** turns the questions around: protected attributes
  usually can't be used at decision time, and the harms are different. Same tools,
  different answers.